In [20]:
%pip install torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import io
import re
import numpy as np
from gensim.models import KeyedVectors

## Загрузка полных векторов

In [2]:
def load_vectors(fname):
    """Загружает текстовые векторы fastText в память (слово -> вектор)."""
    fin = io.open(fname, 'r', encoding='utf-8', newline='\n', errors='ignore')
    n, d = map(int, fin.readline().split())
    data = {}
    for line in fin:
        tokens = line.rstrip().split(' ')
        # Преобразуем строки в float32 — экономим память
        data[tokens[0]] = np.asarray(tokens[1:], dtype='float32')
    return data

In [3]:
print("Загружаю русские эмбеддинги...")
ru_emb = load_vectors('resources/cc.ru.300.vec')
print(f"Русских слов: {len(ru_emb)}")

print("Загружаю польские эмбеддинги...")
pl_emb = load_vectors('resources/cc.pl.300.vec')
print(f"Польских слов: {len(pl_emb)}")

Загружаю русские эмбеддинги...
Русских слов: 2000000
Загружаю польские эмбеддинги...
Польских слов: 2000000


## Загрузка учебного словаря и построение матриц X, Y

In [4]:
def load_train_dict(path):
    pairs = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) != 2:
                continue
            ru_word, pl_word = parts
            if ru_word in ru_emb and pl_word in pl_emb:
                pairs.append((ru_word, pl_word))
    return pairs

train_pairs = load_train_dict('resources/train_dict_ru_pl.txt')

In [5]:
X = np.array([ru_emb[ru] for ru, pl in train_pairs])
Y = np.array([pl_emb[pl] for ru, pl in train_pairs])
print(f"X.shape = {X.shape}, Y.shape = {Y.shape}")

X.shape = (20935, 300), Y.shape = (20935, 300)


## Обучение W через ортогональный Procrustes

In [6]:
def procrustes(X, Y):
    """
    Ортогональное решение Procrustes.
    Находит W, минимизирующую ||W X - Y||_F при W^T W = I.
    """
    # Сингулярное разложение Y^T X
    U, S, Vt = np.linalg.svd(Y.T @ X)
    W = U @ Vt
    return W

W = procrustes(X, Y)
print(f"W.shape = {W.shape}")

# Проверим ортогональность: W^T W ≈ I
print("Ортогональность W:", np.allclose(W.T @ W, np.eye(W.shape[0]), atol=1e-3))

W.shape = (300, 300)
Ортогональность W: True


In [7]:
def translate_word_nn(word, ru_emb, pl_emb, W, topn=5):
    if word not in ru_emb:
        return ["[OOV]"]
    
    x = ru_emb[word]
    transformed = W @ x
    transformed_norm = transformed / np.linalg.norm(transformed)
    
    # Перебираем все польские слова (для простоты; в реальности — FAISS)
    scores = {}
    for pl_word, pl_vec in pl_emb.items():
        pl_vec_norm = pl_vec / (np.linalg.norm(pl_vec) + 1e-10)
        scores[pl_word] = float(np.dot(transformed_norm, pl_vec_norm))
    
    sorted_words = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [w for w, s in sorted_words[:topn]]

# Пример
print(translate_word_nn("кот", ru_emb, pl_emb, W)[0])

kot


## Метрика CSLS (формула 6 из статьи)
CSLS борется с проблемой «хабов» — слов, которые становятся ближайшими соседями для многих.

$CSLS(W x_s, y_t) = 2 cos(W x_s, y_t) − r_T(W x_s) − r_S(y_t)$

In [8]:
def build_csls_cache(ru_emb, pl_emb, W, K=10, max_words=50000, chunk=1000):
    """
    Считает r_T и r_S порциями по chunk строк.
    Полные матрицы схожести (50000x50000 float64 ~ 20 ГБ) не создаются — только float32-порции.
    """
    ru_words = list(ru_emb.keys())[:max_words]
    pl_words = list(pl_emb.keys())[:max_words]

    X_sub = np.asarray([ru_emb[w] for w in ru_words], dtype=np.float32)
    Y_sub = np.asarray([pl_emb[w] for w in pl_words], dtype=np.float32)

    X_sub /= (np.linalg.norm(X_sub, axis=1, keepdims=True) + 1e-10)
    Y_sub /= (np.linalg.norm(Y_sub, axis=1, keepdims=True) + 1e-10)

    # Отображаем русские в польское пространство (в float32)
    W = np.asarray(W, dtype=np.float32)
    WX = X_sub @ W.T
    WX /= (np.linalg.norm(WX, axis=1, keepdims=True) + 1e-10)

    n_ru, n_pl = len(ru_words), len(pl_words)

    # r_T: для каждого русского — среднее сходство с K ближайшими польскими
    r_T = np.empty(n_ru, dtype=np.float32)
    for s in range(0, n_ru, chunk):
        e = min(s + chunk, n_ru)
        sims = WX[s:e] @ Y_sub.T                     # (chunk, n_pl), float32
        top = np.argpartition(sims, n_pl - K, axis=1)[:, -K:]
        r_T[s:e] = np.take_along_axis(sims, top, axis=1).mean(axis=1)

    # r_S: для каждого польского — среднее сходство с K ближайшими русскими
    r_S = np.empty(n_pl, dtype=np.float32)
    for s in range(0, n_pl, chunk):
        e = min(s + chunk, n_pl)
        sims = Y_sub[s:e] @ WX.T
        top = np.argpartition(sims, n_ru - K, axis=1)[:, -K:]
        r_S[s:e] = np.take_along_axis(sims, top, axis=1).mean(axis=1)

    return ru_words, pl_words, WX, Y_sub, r_T, r_S
    
# Переводл с CSLS
def translate_word_csls(word, ru_emb, pl_emb, W,
                        ru_words, pl_words, WX, Y_sub, r_T, r_S,
                        topn=5):
    if word not in ru_emb or word not in ru_words:
        return ["[OOV]"]
    
    i = ru_words.index(word)
    # Сходство с польскими словами
    sims = WX[i] @ Y_sub.T  # (len(pl_words),)
    # CSLS
    csls_scores = 2 * sims - r_T[i] - r_S
    # Топ-n
    idx = np.argsort(-csls_scores)[:topn]
    return [pl_words[j] for j in idx]

In [9]:
import re

def tokenize(text):
    # Простая токенизация: только буквы, цифры, дефис
    return re.findall(r'[а-яёa-z0-9\-]+', text.lower())

def translate_text(text, ru_emb, pl_emb, W,
                   ru_words, pl_words, WX, Y_sub, r_T, r_S,
                   topn=1):
    tokens = tokenize(text)
    translated = []
    for tok in tokens:
        if tok in ru_emb:
            pl_words_top = translate_word_csls(
                tok, ru_emb, pl_emb, W,
                ru_words, pl_words, WX, Y_sub, r_T, r_S, topn=topn
            )
            translated.append(pl_words_top[0] if pl_words_top else tok)
        else:
            # Если слова нет в эмбеддингах — оставляем как есть
            translated.append(tok)
    return " ".join(translated)

In [10]:
# 1. Загрузка
ru_emb = load_vectors('resources/cc.ru.300.vec')
pl_emb = load_vectors('resources/cc.pl.300.vec')

# 2. Словарь
train_pairs = load_train_dict('resources/train_dict_ru_pl.txt')
X = np.array([ru_emb[ru] for ru, pl in train_pairs])
Y = np.array([pl_emb[pl] for ru, pl in train_pairs])

# 3. Procrustes
W = procrustes(X, Y)
print("W обучена")

# 4. CSLS-кэш
print("CSLS кэш")
ru_words, pl_words, WX, Y_sub, r_T, r_S = build_csls_cache(
    ru_emb, pl_emb, W, K=10, max_words=50000
)
print("Кэш готов")

# 5. Перевод текста
text_ru = "Кот сидит на окне и смотрит на улицу."
text_pl = translate_text(
    text_ru, ru_emb, pl_emb, W,
    ru_words, pl_words, WX, Y_sub, r_T, r_S
)
print("RU:", text_ru)
print("PL:", text_pl)

W обучена
CSLS кэш
Кэш готов
RU: Кот сидит на окне и смотрит на улицу.
PL: kot siedzi na oknie i patrzy na ulicę


In [11]:
import os
import re

# ---------------------------------------------------------------
# Токенизация и перевод (используем функции из предыдущих блоков)
# ---------------------------------------------------------------

def tokenize(text):
    """Простая токенизация: только буквы, цифры, дефис. Приводит к нижнему регистру."""
    return re.findall(r'[а-яёa-z0-9\-]+', text.lower())

def translate_text(text, ru_emb, pl_emb, W,
                   ru_words, pl_words, WX, Y_sub, r_T, r_S,
                   topn=1, show_oov=True):
    """
    Пословный перевод текста с русского на польский.
    Возвращает кортеж: (перевод, список_OOV_слов).
    """
    tokens = tokenize(text)
    translated = []
    oov = []  # слова, которых нет в русских эмбеддингах

    for tok in tokens:
        if tok in ru_emb:
            pl_top = translate_word_csls(
                tok, ru_emb, pl_emb, W,
                ru_words, pl_words, WX, Y_sub, r_T, r_S, topn=topn
            )
            translated.append(pl_top[0] if pl_top else tok)
        else:
            translated.append(tok)   # оставляем как есть
            oov.append(tok)

    if show_oov and oov:
        print(f"  [!] Слова вне словаря ({len(oov)}): {', '.join(sorted(set(oov)))}")

    return " ".join(translated), oov


# ---------------------------------------------------------------
# 1. Перевод одного текстового файла
# ---------------------------------------------------------------

def translate_file(input_path, output_path=None, topn=1):
    """
    Читает txt-файл, переводит построчно, сохраняет результат.
    Если output_path=None — сохраняет рядом с исходным с суффиксом '_pl'.
    """
    with open(input_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    print(f"\n=== Перевод файла: {input_path} ===")
    translated_lines = []

    for i, line in enumerate(lines, 1):
        line = line.strip()
        if not line:
            translated_lines.append("")
            continue

        translation, _ = translate_text(
            line, ru_emb, pl_emb, W,
            ru_words, pl_words, WX, Y_sub, r_T, r_S,
            topn=topn
        )
        translated_lines.append(translation)
        # Печатаем прогресс каждые 10 строк, чтобы не засорять вывод
        if i % 10 == 0 or i == len(lines):
            print(f"  обработано строк: {i}/{len(lines)}")

    # Сохранение
    if output_path is None:
        base, ext = os.path.splitext(input_path)
        output_path = f"{base}_pl{ext}"

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write("\n".join(translated_lines))

    print(f"  Готово! Результат сохранён в: {output_path}")
    return output_path


# ---------------------------------------------------------------
# 2. Интерактивный перевод своих сообщений
# ---------------------------------------------------------------

def interactive_translator():
    """Цикл: пользователь вводит русский текст, получает польский перевод."""
    print("\n=== Интерактивный переводчик (RU → PL) ===")
    print("Введите текст на русском и нажмите Enter.")
    print("Для выхода введите: exit / quit / выход\n")

    while True:
        try:
            user_input = input("RU> ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nВыход.")
            break

        if user_input.lower() in ('exit', 'quit', 'выход', ''):
            print("Выход.")
            break

        translation, _ = translate_text(
            user_input, ru_emb, pl_emb, W,
            ru_words, pl_words, WX, Y_sub, r_T, r_S,
            topn=1
        )
        print(f"PL> {translation}\n")

In [12]:
translate_file('examples/вирус.txt', output_path='examples/virus_polish.txt')


=== Перевод файла: examples/вирус.txt ===
  [!] Слова вне словаря (2): 35412, flemming
  [!] Слова вне словаря (3): илиаде, приама, эшнунна
  [!] Слова вне словаря (2): аристею, артемиде
  [!] Слова вне словаря (4): гидрофобию, плиний, средиземноморью, цельс
  [!] Слова вне словаря (2): антарктиду, безконтрольный
  [!] Слова вне словаря (1): пастер
  [!] Слова вне словаря (12): вонна, пастер, пастера, спиномозгового, спиномозговом, теодора, укусившая, францусзкой, эльзаса
  [!] Слова вне словаря (4): адельчи, пастеровского, пуркиньемозжечка, ремлинже
  [!] Слова вне словаря (6): пастеровском, пиван, порт-оф-спэйн, тобаго, тринидад
  обработано строк: 10/21
  [!] Слова вне словаря (6): абсорбирующимися, рабдовирусов, рнк-зависимой-рнк-полимеразы, рнк-содержащий, рнк-содержащих, транспортируемыми
  [!] Слова вне словаря (1): рнк-содержащих
  [!] Слова вне словаря (1): иннервированные
  [!] Слова вне словаря (1): гидрофобией
  [!] Слова вне словаря (1): пастера
  [!] Слова вне словаря (6

'examples/virus_polish.txt'

In [13]:
translate_file('examples/Маша и медведь.txt', output_path='examples/medved.txt')


=== Перевод файла: examples/Маша и медведь.txt ===
  [!] Слова вне словаря (1): машеньку
  [!] Слова вне словаря (1): видит-стоит
  обработано строк: 10/55
  [!] Слова вне словаря (1): машеньку
  [!] Слова вне словаря (1): потужила
  [!] Слова вне словаря (1): машеньке
  обработано строк: 20/55
  [!] Слова вне словаря (1): машеньке
  [!] Слова вне словаря (1): шёл-шёл
  обработано строк: 30/55
  [!] Слова вне словаря (2): шёл-шёл
  обработано строк: 40/55
  [!] Слова вне словаря (2): машеньки, отпирайте
  обработано строк: 50/55
  [!] Слова вне словаря (1): видят-
  [!] Слова вне словаря (2): живёхонька, здоровёхонька
  [!] Слова вне словаря (1): машеньку
  обработано строк: 55/55
  Готово! Результат сохранён в: examples/medved.txt


'examples/medved.txt'

In [14]:
my_text = "Когда-то тебе нужно сделать то За что тебя, мой друг, никто тут не похвалит"
result, _ = translate_text(
    my_text, ru_emb, pl_emb, W,
    ru_words, pl_words, WX, Y_sub, r_T, r_S
)
print("RU:", my_text)
print("PL:", result)

RU: Когда-то тебе нужно сделать то За что тебя, мой друг, никто тут не похвалит
PL: kiedyś tobie wystarczy zrobić to za jednak ciebie mój nawzajem nikt tutaj bo [OOV]


In [15]:
import os
import re
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel

# Отключаем лишние предупреждения
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

MODEL_NAME = "xlm-roberta-base"
print("Загружаю XLM-RoBERTa...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Модель загружена. Устройство: {device}")

c:\Users\reino\Desktop\Учеба\Homework\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Загружаю XLM-RoBERTa...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1229.78it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Модель загружена. Устройство: cpu


In [16]:
def get_contextual_embeddings(sentence, tokenizer, model, device):
    """
    Пропускает предложение через XLM-R и возвращает:
    - список слов
    - список векторов (по одному на слово)
    """
    encoded = tokenizer(
        sentence,
        return_tensors="pt",
        padding=True,
        truncation=True,
        return_offsets_mapping=True
    )
    
    with torch.no_grad():
        outputs = model(
            input_ids=encoded["input_ids"].to(device),
            attention_mask=encoded["attention_mask"].to(device)
        )
    
    token_embeddings = outputs.last_hidden_state[0].cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])
    offsets = encoded["offset_mapping"][0].cpu().numpy()
    
    words = []
    word_vectors = []
    current_word = ""
    current_start = None
    current_vectors = []
    
    for i, (start, end) in enumerate(offsets):
        if start == 0 and end == 0:
            continue
        token_str = tokens[i]
        if token_str.startswith("▁") and current_word:
            words.append(current_word)
            word_vectors.append(np.mean(current_vectors, axis=0))
            current_word = ""
            current_start = None
            current_vectors = []
        if current_start is None:
            current_start = start
        clean_token = token_str.replace("▁", "")
        current_word += clean_token
        current_vectors.append(token_embeddings[i])
    
    if current_word:
        words.append(current_word)
        word_vectors.append(np.mean(current_vectors, axis=0))
    
    return words, np.array(word_vectors)

In [17]:
def get_word_vector_in_context(word, sentence, tokenizer, model, device):
    """Возвращает контекстуальный вектор для слова в предложении."""
    words, vectors = get_contextual_embeddings(sentence, tokenizer, model, device)
    word_lower = word.lower()
    for i, w in enumerate(words):
        if w.lower() == word_lower:
            return vectors[i]
    return None


def choose_best_translation(word_ru, sentence_ru, candidates_pl,
                            tokenizer, model, device,
                            ru_emb, pl_emb):
    """Выбирает лучший перевод среди кандидатов с учётом контекста."""
    if not candidates_pl:
        return None
    if len(candidates_pl) == 1:
        return candidates_pl[0]
    
    ctx_vec_ru = get_word_vector_in_context(
        word_ru, sentence_ru, tokenizer, model, device
    )
    if ctx_vec_ru is None:
        return candidates_pl[0]
    
    ctx_vec_ru_norm = ctx_vec_ru / (np.linalg.norm(ctx_vec_ru) + 1e-10)
    
    best_candidate = candidates_pl[0]
    best_score = -np.inf
    
    for cand in candidates_pl:
        cand_vec = get_word_vector_in_context(
            cand, cand, tokenizer, model, device
        )
        if cand_vec is None:
            continue
        cand_vec_norm = cand_vec / (np.linalg.norm(cand_vec) + 1e-10)
        score = float(np.dot(ctx_vec_ru_norm, cand_vec_norm))
        if score > best_score:
            best_score = score
            best_candidate = cand
    
    return best_candidate

In [18]:
def tokenize(text):
    return re.findall(r'[а-яёa-z0-9\-]+', text.lower())


def translate_text_contextual(text_ru,
                              ru_emb, pl_emb, W,
                              ru_words, pl_words, WX, Y_sub, r_T, r_S,
                              tokenizer, model, device,
                              topn=5, use_context=True):
    """Переводит русский текст на польский."""
    tokens = tokenize(text_ru)
    translated = []
    
    for tok in tokens:
        if tok not in ru_emb:
            translated.append(tok)
            continue
        
        candidates = translate_word_csls(
            tok, ru_emb, pl_emb, W,
            ru_words, pl_words, WX, Y_sub, r_T, r_S,
            topn=topn
        )
        
        if not candidates:
            translated.append(tok)
            continue
        
        if use_context and len(candidates) > 1:
            best = choose_best_translation(
                tok, text_ru, candidates,
                tokenizer, model, device,
                ru_emb, pl_emb
            )
            translated.append(best if best else candidates[0])
        else:
            translated.append(candidates[0])
    
    return " ".join(translated)

In [19]:
def translate_file_contextual(input_path, output_path=None,
                              ru_emb=None, pl_emb=None, W=None,
                              ru_words=None, pl_words=None,
                              WX=None, Y_sub=None, r_T=None, r_S=None,
                              tokenizer=None, model=None, device=None,
                              topn=5, use_context=True,
                              parallel=False):
    """Читает txt-файл, переводит построчно, сохраняет результат."""
    with open(input_path, 'r', encoding='utf-8') as f:
        lines = [l.rstrip('\n') for l in f]
    
    print(f"\n=== Перевод файла: {input_path} ===")
    print(f"Всего строк: {len(lines)}")
    print(f"use_context = {use_context}")   # ← отладка
    
    translated_lines = []
    oov_all = set()
    
    for i, line in enumerate(lines, 1):
        if not line.strip():
            translated_lines.append("")
            continue
        
        translation = translate_text_contextual(
            line,
            ru_emb, pl_emb, W,
            ru_words, pl_words, WX, Y_sub, r_T, r_S,
            tokenizer, model, device,          # ← ЯВНО передаём
            topn=topn,                          # ← ЯВНО
            use_context=use_context             # ← ЯВНО
        )
        
        if parallel:
            translated_lines.append(f"RU: {line}\nPL: {translation}\n")
        else:
            translated_lines.append(translation)
        
        for tok in tokenize(line):
            if tok not in ru_emb:
                oov_all.add(tok)
        
        if i % 5 == 0 or i == len(lines):
            print(f"  обработано строк: {i}/{len(lines)}")
    
    if output_path is None:
        base, ext = os.path.splitext(input_path)
        suffix = "_pl_parallel" if parallel else "_pl"
        output_path = f"{base}{suffix}{ext}"
    
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write("\n".join(translated_lines))
    
    print(f"  Готово! Сохранено в: {output_path}")
    if oov_all:
        print(f"  Слов вне словаря: {len(oov_all)}")
    
    return output_path

In [20]:
FILE_1 = 'examples/вирус.txt'
FILE_2 = 'examples/Маша и медведь.txt'

translate_file_contextual(
    FILE_1,
    ru_emb=ru_emb, pl_emb=pl_emb, W=W,
    ru_words=ru_words, pl_words=pl_words,
    WX=WX, Y_sub=Y_sub, r_T=r_T, r_S=r_S,
    tokenizer=tokenizer, model=model, device=device,
    topn=5,
    use_context=False,   # ← отключаем XLM-R
    parallel=False
)

translate_file_contextual(
    FILE_2,
    ru_emb=ru_emb, pl_emb=pl_emb, W=W,
    ru_words=ru_words, pl_words=pl_words,
    WX=WX, Y_sub=Y_sub, r_T=r_T, r_S=r_S,
    tokenizer=tokenizer, model=model, device=device,
    topn=5, use_context=True, parallel=False
)


=== Перевод файла: examples/вирус.txt ===
Всего строк: 21
use_context = False
  обработано строк: 5/21
  обработано строк: 10/21
  обработано строк: 15/21
  обработано строк: 20/21
  обработано строк: 21/21
  Готово! Сохранено в: examples/вирус_pl.txt
  Слов вне словаря: 53

=== Перевод файла: examples/Маша и медведь.txt ===
Всего строк: 55
use_context = True
  обработано строк: 5/55
  обработано строк: 10/55
  обработано строк: 15/55
  обработано строк: 20/55
  обработано строк: 25/55
  обработано строк: 30/55
  обработано строк: 35/55
  обработано строк: 40/55
  обработано строк: 45/55
  обработано строк: 50/55
  обработано строк: 55/55
  Готово! Сохранено в: examples/Маша и медведь_pl.txt
  Слов вне словаря: 10


'examples/Маша и медведь_pl.txt'

## 6. Учебный и тестовый словари



Из словаря `resources/train_dict_ru_pl.txt` случайно берём **500 пар** для обучения `W`

(seed зафиксирован, чтобы выборка была воспроизводимой). Остальные пары — тестовые,

на них считается Precision@k. Между обучением и оценкой нет пересечения слов.

In [ ]:
import random

def load_pairs(path, ru_emb, pl_emb):
    """Читает словарь 'ru<TAB>pl', оставляет только покрытые эмбеддингами однословные пары."""
    pairs = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip('\n').split('\t')
            if len(parts) != 2:
                continue
            ru, pl = parts[0].strip().lower(), parts[1].strip().lower()
            if ru in ru_emb and pl in pl_emb:
                pairs.append((ru, pl))
    return pairs

def sample_train_test(pairs, n_train=500, seed=42):
    """n_train случайных пар -> обучение, остальные -> тест."""
    rng = random.Random(seed)
    idx = list(range(len(pairs)))
    rng.shuffle(idx)
    train_i = set(idx[:n_train])
    train = [pairs[i] for i in idx[:n_train]]
    test = [p for i, p in enumerate(pairs) if i not in train_i]
    return train, test

all_pairs = load_pairs('resources/train_dict_ru_pl.txt', ru_emb, pl_emb)
train_pairs500, test_pairs = sample_train_test(all_pairs, n_train=500, seed=42)
print(f'Всего пар: {len(all_pairs)} | учебные: {len(train_pairs500)} | тестовые: {len(test_pairs)}')

with open('resources/train_dict_500.txt', 'w', encoding='utf-8') as f:
    for ru, pl in train_pairs500:
        f.write(f'{ru}\t{pl}\n')
print('учебный словарь сохранён: resources/train_dict_500.txt')

## 7. Метрика качества: Precision@k



Для каждой тестовой пары (ru, pl) ищем k соседей среди польского пула по CSLS

и проверяем, попал ли эталонный перевод в топ-k. Слова вне пула (редкие) пропускаем,

поэтому вместе с точностью печатаем число реально оценённых пар.

In [ ]:
def precision_at_k(pairs, ru_words, pl_words, WX, Y_sub, r_T, r_S, ks=(1, 5)):
    """
    Precision@k по словарю пар с использованием CSLS-кэша.
    Возвращает (число оценённых пар, {k: точность}).
    """
    ru_idx = {w: i for i, w in enumerate(ru_words)}
    pl_idx = {w: j for j, w in enumerate(pl_words)}
    ks = tuple(ks)
    hits = {k: 0 for k in ks}
    n_eval = 0
    for ru, pl in pairs:
        i = ru_idx.get(ru)
        j = pl_idx.get(pl)
        if i is None or j is None:
            continue
        sims = WX[i] @ Y_sub.T
        csls = 2 * sims - r_T[i] - r_S
        order = np.argsort(-csls)[:max(ks)]
        n_eval += 1
        for k in ks:
            if j in order[:k]:
                hits[k] += 1
    if n_eval == 0:
        return 0, {}
    return n_eval, {k: hits[k] / n_eval for k in ks}

## 8. Эксперимент 1: fastText + Procrustes + CSLS



`W` обучается на 500 случайных парах, метрика — на остальных.

In [ ]:
X_tr_ft = np.array([ru_emb[r] for r, p in train_pairs500], dtype='float32')
Y_tr_ft = np.array([pl_emb[p] for r, p in train_pairs500], dtype='float32')
W_ft = procrustes(X_tr_ft, Y_tr_ft)

# пул кандидатов для CSLS: топ-50000 частотных слов каждого языка
ru_words_ft, pl_words_ft, WX_ft, Ysub_ft, rT_ft, rS_ft = build_csls_cache(
    ru_emb, pl_emb, W_ft, K=10, max_words=50000
)

n_eval_ft, acc_ft = precision_at_k(test_pairs, ru_words_ft, pl_words_ft,
                                    WX_ft, Ysub_ft, rT_ft, rS_ft)
print(f'fastText: оценено пар {n_eval_ft}/{len(test_pairs)}')
print(f'  Precision@1 = {acc_ft.get(1, 0):.3f}  Precision@5 = {acc_ft.get(5, 0):.3f}')

results_summary = {'fastText + Procrustes + CSLS': acc_ft}

## 9. Эксперимент 2: собственные эмбеддинги Word2Vec



Корпуса **не скачиваются внутри ноутбука**. Сначала один раз запусти скрипт из папки проекта:



```

python download_w2v_corpus.py

```



Он положит тексты в `resources/wiki_ru_full.txt` и `resources/wiki_pl_full.txt`

(готовый датасет `wikimedia/wikipedia` через HuggingFace datasets-server, ~5 МБ на язык).



Ячейка ниже читает эти файлы; если их ещё нет — берёт локальные романы/книги и печатает

предупреждение. Обучающий словарь тот же, что и в эксперименте 1: пара берётся в расчёт,

только если оба слова есть в наших w2v-словарях.

In [ ]:
import os

def load_wiki_corpus(path, min_block=50):
    """Читает скачанный скриптом корпус: статьи разделены пустыми строками."""
    if not (path and os.path.exists(path)):
        return []
    with open(path, encoding='utf-8') as f:
        return [b for b in f.read().split('\n\n') if len(b.strip()) > min_block]

def load_books_dir(cache_dir):
    """Читает заранее скачанные книги Project Gutenberg (если директория есть)."""
    if not os.path.isdir(cache_dir):
        return []
    texts = []
    for name in sorted(os.listdir(cache_dir)):
        if name.endswith('.txt'):
            with open(os.path.join(cache_dir, name), encoding='utf-8', errors='ignore') as f:
                texts.append(f.read())
    return texts

# --- основной корпус: файлы из download_w2v_corpus.py ---
ru_texts = load_wiki_corpus('resources/wiki_ru_full.txt')
pl_texts = load_wiki_corpus('resources/wiki_pl_full.txt')

# --- локальные запасы, если wiki-файлов ещё нет ---
for path in ['data/MasterAndMargarita.txt', 'data/flowers.txt']:
    if os.path.exists(path):
        with open(path, encoding='utf-8', errors='ignore') as f:
            ru_texts.append(f.read())
if not pl_texts:
    pl_texts = load_books_dir('resources/pl_books')
if not pl_texts:
    # самый последний резерв: строки словаря как микро-корпус (очень слабо)
    pl_texts = [l.split('\t')[0] for l in open('resources/pl-ru_wiki.txt', encoding='utf-8') if '\t' in l]

if len(''.join(pl_texts)) < 3_000_000:
    print('ВНИМАНИЕ: польский корпус маленький. Для нормальных метрик сначала выполни:')
    print('    python download_w2v_corpus.py')

def tokenize_uni(text):
    """Токенизация для Word2Vec: русские и польские буквы (с диакритикой), цифры, дефис."""
    return re.findall(r"[a-ząćęłńóśźżjа-яё0-9][a-ząćęłńóśźżjа-яё0-9\-]*", text.lower())

ru_sents = [tokenize_uni(t) for t in ru_texts]
pl_sents = [tokenize_uni(t) for t in pl_texts]
print(f'RU токенов: {sum(map(len, ru_sents)):,} | PL токенов: {sum(map(len, pl_sents)):,}')

### Обучение Word2Vec



Skip-gram (`sg=1`), 300 измерений — как у fastText, чтобы результаты были сопоставимы.

`min_count=3` и `epochs=10` — баланс между качеством и временем на CPU.

Модели кешируются в `resources/w2v_*_v2.model`; переиспользуются только если словарь

достаточно большой, иначе переобучиваются на текущем корпусе.

In [ ]:
from gensim.models import Word2Vec

W2V_PARAMS = dict(vector_size=300, window=5, min_count=3, sg=1, epochs=10, workers=4)

RU_W2V = 'resources/w2v_ru_v2.model'
PL_W2V = 'resources/w2v_pl_v2.model'
def expected_min_vocab(sentences):
    """Ориентировочный минимальный словарь для текущего корпуса (~3% токенов)."""
    return max(20000, int(0.03 * sum(len(s) for s in sentences)))

def train_or_load(model_path, sentences):
    """Загружает модель с диска, если словарь достаточно большой; иначе обучает заново."""
    if os.path.exists(model_path):
        m = Word2Vec.load(model_path)
        if len(m.wv) >= expected_min_vocab(sentences):
            print(f'  {model_path}: загружена, словарь {len(m.wv)}')
            return m
        print(f'  {model_path}: словарь мал ({len(m.wv)}), переобучаю')
    m = Word2Vec(sentences, **W2V_PARAMS)
    m.save(model_path)
    print(f'  {model_path}: обучена, словарь {len(m.wv)}')
    return m

print('Обучаю Word2Vec (RU)...')
model_ru = train_or_load(RU_W2V, ru_sents)
print('Обучаю Word2Vec (PL)...')
model_pl = train_or_load(PL_W2V, pl_sents)

def w2v_to_dict(model):
    """dict word -> np.float32, порядок ключей по частоте (как в .vec fastText)."""
    return {w: model.wv.get_vector(w) for w in model.wv.index_to_key}

ru_emb_w2v = w2v_to_dict(model_ru)
pl_emb_w2v = w2v_to_dict(model_pl)
print(f'Словарь RU w2v: {len(ru_emb_w2v)} | PL w2v: {len(pl_emb_w2v)}')

In [ ]:
# учебные 500 пар фильтруются по покрытию нашими w2v-эмбеддингами
train_w2v = [(r, p) for r, p in train_pairs500 if r in ru_emb_w2v and p in pl_emb_w2v]
test_w2v  = [(r, p) for r, p in test_pairs  if r in ru_emb_w2v and p in pl_emb_w2v][:3000]
print(f'w2v: train={len(train_w2v)}/{len(train_pairs500)}, test={len(test_w2v)}')

# векторы Word2Vec не нормированы, поэтому перед Procrustes приводим строки к длине 1
X_tr_w = np.array([ru_emb_w2v[r] for r, p in train_w2v], dtype='float32')
Y_tr_w = np.array([pl_emb_w2v[p] for r, p in train_w2v], dtype='float32')
X_tr_w /= (np.linalg.norm(X_tr_w, axis=1, keepdims=True) + 1e-10)
Y_tr_w /= (np.linalg.norm(Y_tr_w, axis=1, keepdims=True) + 1e-10)

W_w2v = procrustes(X_tr_w, Y_tr_w)

ru_words_w2v, pl_words_w2v, WX_w, Ysub_w, rT_w, rS_w = build_csls_cache(
    ru_emb_w2v, pl_emb_w2v, W_w2v, K=10, max_words=len(ru_emb_w2v)
)

n_eval_w, acc_w = precision_at_k(test_w2v, ru_words_w2v, pl_words_w2v,
                                 WX_w, Ysub_w, rT_w, rS_w)
print(f'Word2Vec: оценено пар {n_eval_w}/{len(test_w2v)}')
print(f'  Precision@1 = {acc_w.get(1, 0):.3f}  Precision@5 = {acc_w.get(5, 0):.3f}')

results_summary['Word2Vec + Procrustes + CSLS'] = acc_w

## 10. Эксперимент 3: контекстуализированные эмбеддинги (XLM-R)

Метрика для словарных пар без реального контекста — это надстройка над CSLS: берём топ-5 кандидатов из fastText+CSLS и переранжируем их через XLM-RoBERTa (функция `choose_best_translation` из раздела выше). Сравниваем Precision@1 «голый CSLS» и «CSLS + XLM-R rerank» на подмножестве тестовых пар (по умолчанию 300 — из-за инференса на CPU это ~1 минута).

In [ ]:
def eval_contextual_rerank(pairs, ru_words, pl_words, WX, Y_sub, r_T, r_S,
                           topn=5, limit=300):
    """
    P@1 базового CSLS против CSLS + переранжировка XLM-R.
    Эмбеддинги кэшируются, чтобы не прогонять модель на каждом кандидате заново.
    """
    ru_idx = {w: i for i, w in enumerate(ru_words)}
    pl_set = set(pl_words)
    vec_cache = {}

    def xlmr_vec(w):
        if w not in vec_cache:
            vec_cache[w] = get_word_vector_in_context(w, w, tokenizer, model, device)
        return vec_cache[w]

    n = base_hits = ctx_hits = 0
    for ru, pl in pairs:
        if n >= limit:
            break
        i = ru_idx.get(ru)
        if i is None or pl not in pl_set:
            continue
        sims = WX[i] @ Y_sub.T
        order = np.argsort(-(2 * sims - r_T[i] - r_S))[:topn]
        cands = [pl_words[j] for j in order]
        n += 1
        if cands[0] == pl:
            base_hits += 1
        rv = xlmr_vec(ru)
        if rv is None:
            continue
        rv = rv / (np.linalg.norm(rv) + 1e-10)
        best, best_s = cands[0], -np.inf
        for c in cands:
            cv = xlmr_vec(c)
            if cv is None:
                continue
            s = float(np.dot(rv, cv / (np.linalg.norm(cv) + 1e-10)))
            if s > best_s:
                best_s, best = s, c
        if best == pl:
            ctx_hits += 1
    return n, base_hits / max(n, 1), ctx_hits / max(n, 1)

n_ctx, p1_base, p1_ctx = eval_contextual_rerank(
    test_pairs, ru_words_ft, pl_words_ft, WX_ft, Ysub_ft, rT_ft, rS_ft,
    topn=5, limit=300
)
print(f'Оценено пар: {n_ctx}')
print(f'  CSLS (fastText):        P@1 = {p1_base:.3f}')
print(f'  CSLS + XLM-R rerank:    P@1 = {p1_ctx:.3f}')

results_summary['CSLS + XLM-R rerank (subset P@1)'] = {'1': p1_ctx}

## 11. Итоговое сравнение экспериментов

Сводная таблица по всем методам (значения подставляются из `results_summary`).

In [ ]:
def _acc_get(acc, k):
    """Достаёт точность для k, независимо от типа ключа (int или str)."""
    if not isinstance(acc, dict):
        return None
    for key in (k, str(k)):
        if key in acc:
            return acc[key]
    return None

print(f'{"Метод":<35} {"P@1":>8} {"P@5":>8}')
print('-' * 55)
for name, acc in results_summary.items():
    p1 = _acc_get(acc, 1)
    p5 = _acc_get(acc, 5)
    p1s = f'{p1:.3f}' if p1 is not None else '  -  '
    p5s = f'{p5:.3f}' if p5 is not None else '  -  '
    print(f'{name:<35} {p1s:>8} {p5s:>8}')

## 11. Итоговый вывод

1. **fastText + Procrustes + CSLS** — базовый метод статьи, обученный на готовых эмбеддингах с сотнями миллиардов токенов: лучший результат (см. таблицу выше).
2. **Word2Vec + Procrustes + CSLS** — тот же метод, но эмбеддинги обучены на наших нескольких мегабайтах вики-текста. Качество заметно ниже: пространств ещё нет гипотезы изометрии, а редкие слова из учебного словаря (имена собственные) просто не набрали min_count. Это ожидаемый результат малоданных условий; для роста метрик нужен больший корпус: python download_w2v_corpus.py all 20000000 и перезапуск ячеек §9 (модели переобучатся автоматически).
3. **Контекстуализация (XLM-R)** переранжирует топ-k кандидатов CSLS и помогает многозначным словам в реальных текстах; в словарной оценке без контекста её выигрыш ограничен.
4. Все три подхода используют одну и ту же идею: линейное отображение одного эмбеддингового пространства в другое + поиск ближайших соседей по косинусу/CSLS.